In [ ]:
import kagglehub
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
from torch.optim import AdamW
import torch
from tqdm import tqdm
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, get_scheduler


EMBED_DIM = 128
HIDDEN_DIM = 64
MAX_EPOCHS = 15

path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
df = pd.read_csv(os.path.join(path, "Reviews.csv"))

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

In [23]:
df["clean_text"] = df["Text"].str.lower()
df["clean_text"] = df["clean_text"].str.replace(r"<[^>]+>", " ", regex=True)
df["clean_text"] = df["clean_text"].str.replace(r"[^\w\s]", " ", regex=True)

In [ ]:
def tokenize(texts, max_length=256):
    return tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')


In [25]:
from collections import Counter

# 1. Construire le vocabulaire
all_words = " ".join(df["clean_text"]).split()
vocab = {"<PAD>": 0, "<UNK>": 1}
# 22 000 mots sont présents 20 fois ou plus dans les reviews, on ignore les autres
for word, count in Counter(all_words).most_common(20000):
    vocab[word] = len(vocab)

vocab_size = len(vocab)
# 95% des reviews ont moins de 222 tokens

df["tokenized"] = df["clean_text"].apply(tokenize, vocab=vocab, maxlen=222)

In [26]:
# Sampling dataset
# Train = 70% / Validation = 15% / Test = 15%
train_df = df.sample(frac=0.7, random_state=42)
#train_df = df.sample(n=100, random_state=42)
temp_df = df.drop(train_df.index)
#val_df = temp_df.sample(n=20, random_state=42)
val_df = temp_df.sample(frac=0.15, random_state=42)
temp_df = temp_df.drop(val_df.index)
test_df = temp_df.sample(frac=0.15, random_state=42)

In [27]:
class ReviewDataset(Dataset):
    def __init__(self, df):
        super().__init__()
        self.data = df[["tokenized", "Score"]].values
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, score = self.data[idx]
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(score -1, dtype=torch.long)

In [ ]:
dataloader_train = DataLoader(ReviewDataset(train_df), batch_size=32, shuffle=True)
dataloader_val = DataLoader(ReviewDataset(val_df), batch_size=32)
dataloader_test = DataLoader(ReviewDataset(test_df), batch_size=32)

device = torch.device("cpu")
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  device = torch.device("mps")

print(device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=MAX_EPOCHS*len(dataloader_train))

cuda
NVIDIA GeForce RTX 4060 Ti


In [ ]:
previous_val_loss = 1
for epoch in range(MAX_EPOCHS):
    # Entrainement
    model.train()
    for X_batch, Y_batch in tqdm(dataloader_train, desc=f"Epoch {epoch+1} - Training"):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()
        predicted = model(X_batch)
        loss = criterion(predicted, Y_batch)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for X_batch, Y_batch in tqdm(dataloader_val, desc=f"Epoch {epoch+1} - Validation"):
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
            predicted = model(X_batch)
            predicted_value = predicted.argmax(dim=1)
            correct += (Y_batch == predicted_value).sum().item()
            total += Y_batch.size(0)
            val_loss += criterion(predicted, Y_batch).item()
    val_accuracy = correct / total
    val_loss /= len(dataloader_val)

    scheduler.step()
    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | val_loss: {val_loss:.4f} | val_accuracy: {val_accuracy:.4f} | lr: {optimizer.param_groups[0]['lr']:.6f}")
#    if (previous_val_loss - val_loss) < 0.005:
#        break
    previous_val_loss = val_loss

Epoch 1 - Validation: 100%|██████████| 800/800 [00:02<00:00, 276.52it/s]


Epoch 1/15 | val_loss: 0.5939 | val_accuracy: 0.7760 | lr: 0.001000


Epoch 2 - Validation: 100%|██████████| 800/800 [00:02<00:00, 282.59it/s]


Epoch 2/15 | val_loss: 0.5530 | val_accuracy: 0.7958 | lr: 0.001000


Epoch 3 - Validation: 100%|██████████| 800/800 [00:02<00:00, 284.68it/s]


...

Epoch 13/15 | val_loss: 0.6618 | val_accuracy: 0.8284 | lr: 0.000500


Epoch 14 - Validation: 100%|██████████| 800/800 [00:02<00:00, 282.28it/s]


Epoch 14/15 | val_loss: 0.7012 | val_accuracy: 0.8234 | lr: 0.000500


Epoch 15 - Validation: 100%|██████████| 800/800 [00:02<00:00, 282.99it/s]

Epoch 15/15 | val_loss: 0.7377 | val_accuracy: 0.8261 | lr: 0.000500


In [31]:
torch.save(model.state_dict(), "../model.pt")